# [실습] 예측 원점 5개 평가표 — 학습 폴드 안의 전처리

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 오늘의 후보 4개와 평가 설계

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">시계열 예측 · 넷째 날</div>
<h1 style="color:#000000;margin:10px 0">시계열 교차 검증 — 예측 원점 5개 평가표와 학습 폴드 안의 전처리</h1>
<div style="font-size:1.05em;color:#656565">예측 원점 5개에서 두 모형과 베이스라인을 같은 조건으로 채점하고, 학습 폴드 안에서 전처리를 끝낸 교차 검증 값을 만듭니다</div>
</div>

**오늘의 질문** 둘째 날 ARIMA 와 셋째 날 SARIMA 를 운영에 투입해도 될까요? 11월 16일 한 주의 RMSE 로는 답하지 않습니다. **예측 원점(forecast origin)** 5개의 평균 RMSE 를 계절 나이브 168 과 비교해 답합니다. 계절 나이브 168 은 1주일 전 같은 시각 값을 그대로 예측으로 쓰는 규칙이고, 이 모듈의 **베이스라인(baseline)** 입니다.

1. 맨 위 설치 셀을 한 번 실행합니다. 런타임 유형은 CPU 로 둡니다.
2. 「오늘의 준비」 셀을 실행해 두고, 출력을 기다리는 동안 개념 확인 두 문항을 풉니다.
3. 노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다.

In [ ]:
# 실습 준비 - 패키지 설치
!pip install -q "numpy>=1.26" "pandas>=2.0" "matplotlib>=3.7" "statsmodels==0.14.6" "scikit-learn==1.8.0"

import os, json, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.font_manager as fm
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings; warnings.filterwarnings("ignore")

# 한글 폰트 — Colab 이면 나눔고딕을 설치한 뒤 그 ttf 를 폰트 목록에 **직접** 등록한다.
#  설치는 디스크에 파일만 만든다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  넣어 줘야 이름이 보인다(넣지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나온다).
import glob, shutil, subprocess
if shutil.which("apt-get"):
    subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):   # 설치 여부와 무관하게 한 번
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150       # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 오늘 그림 전체가 함께 쓰는 색 — 베이스라인=남색, 관측=파랑, 강조=하늘색, 문맥=회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 오늘 그림 모두 이 값으로 고정하고 높이만 그림마다 조절한다
SEED = 42        # 「오늘의 준비」 셀의 KFold(shuffle=True) 가 쓰는 시드 — 고정해야 표 2 의 숫자가 재현된다


def datefmt(ax, fmt="%m-%d", n=5):
    """시간 축 눈금 개수와 표기를 잡아 날짜 라벨이 겹치지 않게 한다"""
    ax.xaxis.set_major_locator(mdates.AutoDateLocator(maxticks=n))
    ax.xaxis.set_major_formatter(mdates.DateFormatter(fmt))


def rmse(a, b):
    """평균 제곱근 오차 — 큰 오차가 제곱되어 더 크게 반영된다"""
    return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))


def mae(a, b):
    """평균 절대 오차 — 오차의 크기를 그대로 평균한다"""
    return float(np.mean(np.abs(np.asarray(a, dtype=float) - np.asarray(b, dtype=float))))


print("사용 폰트:", FONT, "| 색 팔레트 준비 완료 — 다음 셀이 오늘의 채점 셀을 올립니다")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 받아 실행 결과에 채점 결과를 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    한글이 □ 로 깨진다(2026-09-09 Colab 실사고).
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 줄들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        plt.rcParams["font.family"] = _font
        plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(json.loads(base64.b64decode(s).decode("utf-8")))


# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
               "eyJtMSI6eyJ2YWx1ZXMiOnsi7IWL7Ke4IOuCoCBTQVJJTUEgNeyjvCDtj4nqt6AgUk1TRSI6WzM4NC42LDM4NS42LCLrjIAiXSwi"
               "6rOE7KCIIOuCmOydtOu4jCAxNjggNeyjvCDtj4nqt6AgUk1TRSI6WzM2My4zLDM2My44LCLrjIAiXX0sInBsb3QiOnsia2luZCI6"
               "ImxpbmUiLCJuX3NlcmllcyI6MywibWluX3BvaW50cyI6Mywibl9wb2ludHMiOls1LDVdLCJ4X3JhbmdlIjpbMSw1XSwi7LaV6528"
               "67KoIjp0cnVlfSwicmVmIjp7IuuRmOynuCDrgqAgQVJJTUEiOls0MDIuNDEsNTQ1LjIsNTA1LjY3LDQwMy45MiwyNzEuMjJdLCLs"
               "hYvsp7gg64KgIFNBUklNQSI6WzM0NS4zOSwzNjguMSw1MDguMDUsMzE3LjY3LDM4Ni4wOF0sIuqzhOygiCDrgpjsnbTruIwgMTY4"
               "IjpbMzA2LjUyLDQxOC4xOCw0NTcuMDcsNDMwLjcyLDIwNS4yNV19fSwibTIiOnsidmFsdWVzIjp7Iuumv+yngCA167aE7ZWgIO2P"
               "ieq3oCBSTVNFIjpbMzIyLjAsMzIzLjAsIuuMgCJdLCLrtoTtlaAgMSDsiqTsvIDsnbzrn6wg6riw7JioIO2Pieq3oCI6WzE4Ljcs"
               "MTguOSwiwrBDIl19LCJwbG90Ijp7ImtpbmQiOiJiYXIiLCJuIjpbNSw1XSwieW1heCI6WzQ5NS4wLDUxNS4wXSwi7LaV652867Ko"
               "Ijp0cnVlfSwicmVmIjp7Iuu2hO2VoOuzhCI6WzI3OS40OCwzMTAuODUsNTA0LjgzLDI0My4wNCwyNzQuMzddfX19")
_REVEAL = {                                # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
    "m1": ("8J+UkyDrr7jshZggMSDsmYTshLHrs7guIEFJIOy9lOuTnCDspIQg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qUIOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L"
               "2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuTpOyWtOyguCDsnojripQgZGYsIE9SSUdJTlMsIEZJVF9TVEFSVCwgT1JERVJTLCBmaXRfc2FyaW1h"
               "eCwgcm1zZSDrpbwg6re464yA66GcIOyTsOqzoCDsg4jroZwg66eM65Ok7KeA64qUIOunkOyVhCDspJguIE9SSUdJTlMg7J2YIOyYiOy4oQogICAg7JuQ7KCQIDXqsJzrpbwg7LCo66GA66"
               "GcIOuPjOuptOyEnCDtlZnsirUg6rWs6rCE7J2AIEZJVF9TVEFSVCDrtoDthLAg6re4IOybkOygkOq5jOyngCBkZlsnZGVtYW5kJ10g66GcIOyemOudvCDspJguIE9SREVSUyDsnZgg65GQ"
               "IOqzhOygiCDssKjsiJjroZwgZml0X3NhcmltYXgg66W8CiAgICDsm5DsoJDrp4jri6Qg64uk7IucIOu2gOultOqzoCwg7JuQ7KCQIDHsi5zqsIQg65Kk67aA7YSwIDE2OOyLnOqwhOydhC"
               "DtlZwg67KI7JeQIOyYiOy4oe2VtCDqt7ggMTY47Iuc6rCEIOyLpOy4oeqzvOydmCBSTVNFIOulvCDqtaztlbQg7KSYLiDqsJnsnYAgMTY47Iuc6rCE7JeQ7IScIDHso7zsnbwg7KCECiAg"
               "ICDqsJLsnYQg6re464yA66GcIOyTsOuKlCDqs4TsoIgg64KY7J2067iMIDE2OCDsnZggUk1TRSDrj4Qg6rWs7ZW0IOykmC4g7JuQ7KCQ66eI64ukIO2VnCDspITslKkg7IS4IOqwkuydhC"
               "DssI3qs6AsIOyEuCDrqqjtmJXsnZggNeyjvCDtj4nqt6DsnYQgIuyFi+ynuCDrgqAgU0FSSU1BIDXso7wg7Y+J6regCiAgICBSTVNFID0gKOyIq+yekCkg64yAIiDtmJXtg5zroZwg7IaM"
               "7IiYIOuRmOynuCDsnpDrpqzquYzsp4Ag7ZWcIOykhOyUqSBwcmludCDtlbQg7KSYLiDqt7jrnpjtlITripQg7ZWcIOyepeunjCwg6rCA66Gc7LaV7J2EIOyYiOy4oSDsm5DsoJAg67KI7Z"
               "i4IDHrtoDthLAgNeq5jOyngOuhnCDrkZDqs6Ag7IS4CiAgICDrqqjtmJXsnZgg6re4IOyjvCBSTVNFIOulvCDsoJDsnYQg7J207J2AIOyEoCDshLgg6rCc66GcIOq3uOumrOqzoCDrspTr"
               "oYDsmYAg65GQIOy2lSDrnbzrsqjsnYQg64us7JWEIOykmC4KCiAgICByb3dzID0gW10KICAgIGZvciBrLCBvcmlnaW4gaW4gZW51bWVyYXRlKE9SSUdJTlMsIDEpOgogICAgICAgIHlfZm"
               "l0ID0gZGYubG9jW0ZJVF9TVEFSVDpvcmlnaW4sICJkZW1hbmQiXSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMg7ZWZ7Iq17J2AIDnsm5QgMeydvOu2gO2EsCDsm5Ds"
               "oJDquYzsp4AKICAgICAgICB3ZWVrID0gcGQuZGF0ZV9yYW5nZShvcmlnaW4gKyBwZC5UaW1lZGVsdGEoaG91cnM9MSksIHBlcmlvZHM9MTY4LCBmcmVxPSJoIikgICAjIOybkOygkCDrkq"
               "QgMTY47Iuc6rCECiAgICAgICAgeV93ZWVrID0gZGYubG9jW3dlZWssICJkZW1hbmQiXS52YWx1ZXMKICAgICAgICByb3cgPSB7IuybkOygkCI6IGt9CiAgICAgICAgZm9yIG5hbWUsIHNl"
               "YXMgaW4gT1JERVJTLml0ZW1zKCk6CiAgICAgICAgICAgIGZjID0gZml0X3NhcmltYXgoeV9maXQsIHNlYXMpLmdldF9mb3JlY2FzdChzdGVwcz0xNjgpLnByZWRpY3RlZF9tZWFuLnZhbH"
               "VlcyAgICMg7JuQ7KCQ66eI64ukIOyerOygge2VqQogICAgICAgICAgICByb3dbbmFtZV0gPSBybXNlKHlfd2VlaywgZmMpCiAgICAgICAgcm93WyLqs4TsoIgg64KY7J2067iMIDE2OCJd"
               "ID0gcm1zZSh5X3dlZWssIGRmWyJkZW1hbmQiXS5zaGlmdCgxNjgpLmxvY1t3ZWVrXS52YWx1ZXMpICAgIyAx7KO87J28IOyghCDqsJnsnYAg7Iuc6rCBIOqwkgogICAgICAgIHJvd3MuYX"
               "BwZW5kKHJvdykKICAgICAgICBwcmludChmIuybkOygkCB7a30ge29yaWdpbjolbS0lZCAlSOyLnH0gIO2VmeyKtSB7bGVuKHlfZml0KTosfeyLnOqwhCAgIOuRmOynuCDrgqAgQVJJTUEg"
               "e3Jvd1sn65GY7Ke4IOuCoCBBUklNQSddOi4yZn0g64yAIgogICAgICAgICAgICAgIGYiIMK3IOyFi+ynuCDrgqAgU0FSSU1BIHtyb3dbJ+yFi+ynuCDrgqAgU0FSSU1BJ106LjJmfSDrjI"
               "Agwrcg6rOE7KCIIOuCmOydtOu4jCAxNjgge3Jvd1sn6rOE7KCIIOuCmOydtOu4jCAxNjgnXTouMmZ9IOuMgCIpCiAgICB0YWJsZSA9IHBkLkRhdGFGcmFtZShyb3dzKQogICAgcHJpbnQo"
               "ZiLrkZjsp7gg64KgIEFSSU1BIDXso7wg7Y+J6regIFJNU0UgPSB7dGFibGVbJ+uRmOynuCDrgqAgQVJJTUEnXS5tZWFuKCk6LjJmfSDrjIAiKQogICAgcHJpbnQoZiLshYvsp7gg64KgIF"
               "NBUklNQSA17KO8IO2Pieq3oCBSTVNFID0ge3RhYmxlWyfshYvsp7gg64KgIFNBUklNQSddLm1lYW4oKTouMmZ9IOuMgCIpCiAgICBwcmludChmIuqzhOygiCDrgpjsnbTruIwgMTY4IDXs"
               "o7wg7Y+J6regIFJNU0UgPSB7dGFibGVbJ+qzhOygiCDrgpjsnbTruIwgMTY4J10ubWVhbigpOi4yZn0g64yAIikKCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KEZJR1"
               "9XICogMC44LCAzLjQpKQogICAgZm9yIG5hbWUsIGNvbG9yIGluIFsoIuuRmOynuCDrgqAgQVJJTUEiLCBDT0xbIuyXsO2ajOyDiSJdKSwgKCLshYvsp7gg64KgIFNBUklNQSIsIENPTFsi"
               "7ZWY64qY7IOJIl0pLAogICAgICAgICAgICAgICAgICAgICAgICAoIuqzhOygiCDrgpjsnbTruIwgMTY4IiwgQ09MWyLrgqjsg4kiXSldOgogICAgICAgIGF4LnBsb3QodGFibGVbIuybkO"
               "ygkCJdLCB0YWJsZVtuYW1lXSwgbWFya2VyPSJvIiwgY29sb3I9Y29sb3IsIGx3PTEuOCwgbGFiZWw9bmFtZSkKICAgIGF4LnNldF94dGlja3MocmFuZ2UoMSwgNikpCiAgICBheC5zZXRf"
               "eGxhYmVsKCLsmIjsuKEg7JuQ7KCQIikKICAgIGF4LnNldF95bGFiZWwoIuq3uCDso7zsnZggUk1TRSAo64yAKSIpCiAgICBheC5sZWdlbmQoKQogICAgcGx0LnNob3coKQoK7ZW07ISdCg"
               "otIOyYiOyDgSDshYsg6rCA7Jq0642wIOuLteydgCDrkZAg67KI7Ke47J6F64uI64ukLiDsm5DsoJAgNeqwnCDtj4nqt6AgUk1TRSDripQg7IWL7Ke4IOuCoCBTQVJJTUEgMzg1LjA2IOuM"
               "gCwg65GY7Ke4IOuCoCBBUklNQSA0MjUuNjgg64yA7J6F64uI64ukLiDrsqDsnbTsiqTrnbzsnbjsnbgg6rOE7KCIIOuCmOydtOu4jCAxNjgg7J2AIDM2My41NSDrjIDsnoXri4jri6QuIO"
               "uRkCDrqqjtmJUg66qo65GQIOuyoOydtOyKpOudvOyduOuztOuLpCDsu6TshJwg7Yis7J6F7J2EIOuFvOydmO2VoCDsobDqsbTsl5Ag64u/7KeAIOyViuyKteuLiOuLpC4KLSDtlZwg7KO8"
               "7J2YIOyInOychOuKlCDsm5DsoJDsl5Ag65Sw6528IOuwmOuMgOqwgCDrkKnri4jri6QuIOyFi+ynuCDrgqAgU0FSSU1BIOuKlCDsm5DsoJAgNeqwnCDqsIDsmrTrjbAgM+qzsygxMOyblC"
               "AxOeydvCDCtyAxMOyblCAyNuydvCDCtyAxMeyblCA57J28KeyXkOyEnCDrkZjsp7gg64KgIEFSSU1BIOuztOuLpCDsnpHslZjqs6AsIDEx7JuUIDLsnbzsnYAgNTA4LjA1IOuMgOyZgCA1"
               "MDUuNjcg64yA66GcIOqxsOydmCDqsJnslZjsirXri4jri6QuIOyEuCDrqqjtmJUg6rCA7Jq0642wIOqwgOyepSDsnpHsnYAg6rCS7J2AIOqzhOygiCDrgpjsnbTruIwgMTY4IOydtCAz6r"
               "OzLCDshYvsp7gg64KgIFNBUklNQSDqsIAgMuqzs+yXkOyEnCDrsJvslZjsirXri4jri6QuIOyWtOuKkCDso7zsl5Drgpgg7J206riw64qUIOuqqO2YleydtCDsl4bslrTshJwsIO2VnCDs"
               "o7zrp4wg67O06rOgIOqzoOultOuptCDqt7gg7KO87JeQIOuUsOudvCDqsrDroaDsnbQg67CU64Cd64uI64ukLgotIO2dlO2VnCDsi6TsiJgg7ZWY64KY64qUIO2VmeyKtSDquLjsnbTrpb"
               "wgMSw4NDjsi5zqsITsnLzroZwg6rOg7KCV7ZWcIOyxhCDsm5DsoJDrp4wg7Jiu6riw64qUIOqyg+yeheuLiOuLpC4g6re465+s66m0IOyFi+ynuCDrgqAgU0FSSU1BIO2Pieq3oOydtCAz"
               "NzQuODcg64yA66GcIOuLrOudvOynkeuLiOuLpC4g7Ja064qQIOyqveuPhCDti4DrprAg7ISk6rOE64qUIOyVhOuLiOyngOunjCDri6Trpbgg7Iuk7ZeY7J206528LCDtj4nqsIDtkZzsl5"
               "Ag7ZWZ7Iq1IOq1rOqwhOydhCDsoIHsp4Ag7JWK7Jy866m0IOuRkCDqsJLsnYQg6rCZ7J2AIOyLpO2XmOydmCDqsJLsnLzroZwg7J296rKMIOuQqeuLiOuLpC4KLSDtnZTtlZwg7Iuk7IiY"
               "IOuRmOydgCAxMeyblCAxNuydvCDsm5DsoJDsl5DshJwg7ZWcIOuyiCDsoIHtlantlZwg66qo7ZiV7Jy866GcIOuNlCDsnbTrpbgg7JuQ7KCQ65Ok7J2YIOyjvOulvCDsmIjsuKHtlZjrip"
               "Qg6rKD7J6F64uI64ukLiDqt7gg66qo7ZiV7J2YIOqzhOyImOyXkOuKlCDssYTsoJDtlaAg7KO87J2YIOyLpOy4oeydtCDsnbTrr7gg65Ok7Ja0IOyeiOyWtCDriITsiJjqsIAg65Cp64uI"
               "64ukLiDsm5DsoJDrp4jri6Qg6re4IOybkOygkOq5jOyngOydmCDsi6TsuKHsnLzroZzrp4wg64uk7IucIOygge2Vqe2VtOyVvCDsmrTsmIHqs7wg6rCZ7J2AIOyhsOqxtOyeheuLiOuLpC"
               "4KLSDsnbTrsogg7KO87JeQIO2VoCDsnbw6IOyatOyYgSDsmIjsuKHsnYAg6rOE7KCIIOuCmOydtOu4jCAxNjgg66GcIOuDheuLiOuLpC4g7JuQ7KCQIDXqsJzrs4QgUk1TRSDsmYAg7Y+J"
               "6regLCDtlZnsirUg6rWs6rCEKDnsm5QgMeydvOu2gO2EsCDsm5DsoJDquYzsp4ApLCDsmIjsuKEg6riw6rCEIDE2OOyLnOqwhOydhCDtlZwg7ZGc66GcIOyggeyWtCDsmrTsmIHtjIDsl5"
               "Ag67O064OF64uI64ukLiDsg4gg7ZuE67O06rCAIOyDneq4sOuptCDqsJnsnYAg7JuQ7KCQIDXqsJzroZwg7LGE7KCQ7ZW0IOydtCDtkZzsl5Ag7ZWcIO2WieyUqSDrjZTtlanri4jri6Qu"
               "CgrsvZTrk5zqsIAg64u17J2EIOunjOuTnOuKlCDrsKnsi50uIGBkZi5sb2NbRklUX1NUQVJUOm9yaWdpbiwgImRlbWFuZCJdYCDtlZwg7KSE7J20IO2VmeyKtSDqtazqsITsnYQg7JuQ7K"
               "CQ7JeQ7IScIOuBiuqzoCwgYHBkLmRhdGVfcmFuZ2Uob3JpZ2luICsgMeyLnOqwhCwgcGVyaW9kcz0xNjgpYCDsnbQg7LGE7KCQ7ZWgIDE2OOyLnOqwhOydhCDsm5DsoJAg65Kk7JeQ7ISc"
               "IOqzoOumheuLiOuLpC4g7ZWZ7Iq1IOyLnOyekeydhCA57JuUIDHsnbzroZwg65GQ7JeI7Jy866+A66GcIOybkOygkOydtCDtlZwg7KO8IOyYruqyqCDqsIgg65WM66eI64ukIO2VmeyKtS"
               "DquLjsnbTqsIAgMTY47Iuc6rCE7JSpIOuKmOyWtOuCqeuLiOuLpC4K"),
    "m2": ("8J+UkyDrr7jshZggMiDsmYTshLHrs7guIEFJIOy9lOuTnCDspIQg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U65Oc64qUIOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L"
               "2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg7J2066+4IOunjOuTpOyWtOyguCDsnojripQgWF9jdiwgeV9jdiwgVEVNUF9DT0wsIFRpbWVTZXJpZXNTcGxpdCwgbWFr"
               "ZV9waXBlbGluZSwgU3RhbmRhcmRTY2FsZXIsIFJpZGdlLCBybXNlIOulvCDqt7jrjIDroZwg7JOw6rOgIOuLpOyLnAogICAg66eM65Ok7KeAIOunkOyVhCDspJguIFRpbWVTZXJpZXNTcG"
               "xpdChuX3NwbGl0cz01LCB0ZXN0X3NpemU9MTY4KSDroZwgWF9jdiDrpbwg64KY64iE6rOgLCDrtoTtlaDrp4jri6QgbWFrZV9waXBlbGluZShTdGFuZGFyZFNjYWxlcigpLAogICAgUmlk"
               "Z2UoYWxwaGE9MTAwKSkg7J2EIOq3uCDrtoTtlaDsnZgg7ZWZ7Iq1IO2PtOuTnCDtlonsnLzroZzrp4wgZml0IO2VnCDrkqQg6rKA7KadIO2PtOuTnCBSTVNFIOulvCDqtaztlbQg7KSYLi"
               "DrtoTtlaDrp4jri6Qg7ZWZ7Iq1IOq4uOydtOyZgCBSTVNFIOulvCDtlZwg7KSE7JSpIOywjeqzoCwKICAgICLrpr/sp4AgNeu2hO2VoCDtj4nqt6AgUk1TRSA9ICjsiKvsnpApIOuMgCIg"
               "7JmAICLrtoTtlaAgMSDsiqTsvIDsnbzrn6wg6riw7JioIO2Pieq3oCA9ICjsiKvsnpApIMKwQyIg66W8IOyGjOyImCDrkZjsp7gg7J6Q66as66GcIO2VnCDspITslKkgcHJpbnQg7ZW0IO"
               "ykmC4g6riw7JioIO2Pieq3oOydgAogICAg67aE7ZWgIDEg7YyM7J207ZSE65287J247J2YIOyKpOy8gOydvOufrCBtZWFuXyDsl5DshJwgVEVNUF9DT0wg67KIIOyXtOydhCDqurzrgrTq"
               "s6AsIOqwmeydgCDspIQg6rSE7Zi47JeQIOyghOyytCAyLDAxNuyLnOqwhOydmCDquLDsmKgg7Y+J6reg64+EIOyggeyWtCDspJguIOq3uOuemO2UhOuKlCDtlZwKICAgIOyepeunjCwg67"
               "aE7ZWgIDHvvZ41IOydmCDqsoDspp0g7Y+065OcIFJNU0Ug66W8IOunieuMgCDri6TshK8g6rCc66GcIOq3uOumrOqzoCDrkZAg7LaVIOudvOuyqOydhCDri6zslYQg7KSYLgoKICAgIHNw"
               "bGl0dGVyID0gVGltZVNlcmllc1NwbGl0KG5fc3BsaXRzPTUsIHRlc3Rfc2l6ZT0xNjgpICAgICAgIyDqsoDspp0g7Y+065OcIDE2OOyLnOqwhCA16rCcID0gMTDsm5QgMjDsnbzrtoDthL"
               "Ag66eI7KeA66eJIDXso7wKICAgIGZvbGRfcm1zZSwgcGlwZXMgPSBbXSwgW10KICAgIGZvciBrLCAodHIsIHRlKSBpbiBlbnVtZXJhdGUoc3BsaXR0ZXIuc3BsaXQoWF9jdiksIDEpOgog"
               "ICAgICAgIHBpcGUgPSBtYWtlX3BpcGVsaW5lKFN0YW5kYXJkU2NhbGVyKCksIFJpZGdlKGFscGhhPTEwMCkpLmZpdChYX2N2W3RyXSwgeV9jdlt0cl0pICAgIyDsiqTsvIDsnbzrn6zrj4"
               "Qg7ZWZ7Iq1IO2PtOuTnOuhnOunjCBmaXQKICAgICAgICBmb2xkX3Jtc2UuYXBwZW5kKHJtc2UoeV9jdlt0ZV0sIHBpcGUucHJlZGljdChYX2N2W3RlXSkpKQogICAgICAgIHBpcGVzLmFw"
               "cGVuZChwaXBlKQogICAgICAgIHByaW50KGYi67aE7ZWgIHtrfSAg7ZWZ7Iq1IHtsZW4odHIpOix97Iuc6rCEICDqsoDspp0g7Y+065OcIFJNU0Uge2ZvbGRfcm1zZVstMV06LjJmfSDrjI"
               "AiKQogICAgdGVtcF9tdTEgPSBwaXBlc1swXS5uYW1lZF9zdGVwc1sic3RhbmRhcmRzY2FsZXIiXS5tZWFuX1tURU1QX0NPTF0KICAgIHByaW50KGYi66a/7KeAIDXrtoTtlaAg7Y+J6reg"
               "IFJNU0UgPSB7bnAubWVhbihmb2xkX3Jtc2UpOi4yZn0g64yAIikKICAgIHByaW50KGYi67aE7ZWgIDEg7Iqk7LyA7J2865+sIOq4sOyYqCDtj4nqt6AgPSB7dGVtcF9tdTE6LjJmfSDCsE"
               "MgICAo7KCE7LK0IDIsMDE27Iuc6rCEIHtYX2N2WzosIFRFTVBfQ09MXS5tZWFuKCk6LjJmfSDCsEMpIikKCiAgICBmaWcsIGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KEZJR19XICog"
               "MC44LCAzLjQpKQogICAgYXguYmFyKHJhbmdlKDEsIDYpLCBmb2xkX3Jtc2UsIGNvbG9yPUNPTFsi7ZWY64qY7IOJIl0sIHdpZHRoPTAuNikKICAgIGF4LnNldF94dGlja3MocmFuZ2UoMS"
               "wgNikpCiAgICBheC5zZXRfeGxhYmVsKCLrtoTtlaAiKQogICAgYXguc2V0X3lsYWJlbCgi6rKA7KadIO2PtOuTnCBSTVNFICjrjIApIikKICAgIHBsdC5zaG93KCkKCu2VtOyEnQoKLSDs"
               "mIjsg4Eg7IWLIOqwgOyatOuNsCDri7XsnYAg7LKrIOuyiOynuOyeheuLiOuLpC4g67aE7ZWgIDEg7Iqk7LyA7J2865+s7J2YIOq4sOyYqCDtj4nqt6DsnYAgMTguNzkgwrBDIOydtOqzoC"
               "wgOeyblCAx7J28772eMTHsm5QgMjPsnbwg7KCE7LK0IO2Pieq3oOydgCAxNC45MiDCsEMg7J6F64uI64ukLiDrtoTtlaAgMSDsnZgg7JiI7LihIOybkOygkOyduCAxMOyblCAxOeydvCAy"
               "M+yLnOyXkOuKlCAxMeyblOydmCDrgq7snYAg6riw7Jio7J2EIOyVjCDsiJgg7JeG7Jy864uILCDrtoTtlaAgMSDsnbQg7JO4IOyImCDsnojripQg7Y+J6reg7J2AIDE4Ljc5IMKwQyDsno"
               "Xri4jri6QuCi0g7Y+J6regIFJNU0Ug64qUIDMyMi41MiDrjIDsnbTqs6AsIOu2hO2VoOuniOuLpCAyNzkuNDggwrcgMzEwLjg1IMK3IDUwNC44MyDCtyAyNDMuMDQgwrcgMjc0LjM3IOuM"
               "gOyeheuLiOuLpC4g6rKA7KadIO2PtOuTnCA16rCc64qUIOuvuOyFmCAxIOydmCDsmIjsuKEg7JuQ7KCQIDXqsJzqsIAg7LGE7KCQ7ZWcIDXso7zsmYAg6rCZ7J2AIOyjvOyeheuLiOuLpC"
               "4KLSDtnZTtlZwg7Iuk7IiYIO2VmOuCmOuKlCDsiqTsvIDsnbzrn6zrpbwgWF9jdiDsoITssrTroZwg7ZWcIOuyiCBmaXQg7ZWcIOuSpCDrgpjriITripQg6rKD7J6F64uI64ukLiDrtoTt"
               "laAgMSDsnZgg6riw7JioIO2Pieq3oOydtCAxNC45MiDCsEMg6rCAIOuQmOqzoCwg7Y+J6regIFJNU0Ug64qUIDMyMi43NyDrjIDroZwgMC4yNSDrjIDrp4wg64us65287KeR64uI64ukLi"
               "Dsp4DtkZzqsJLsnLzroZzripQg6rGw7J2YIOuTnOufrOuCmOyngCDslYrripQg64iE7IiY6528LCDsiqTsvIDsnbzrn6zsnZggbWVhbl8g7J2EIOywjeyWtCDtmZXsnbjtlanri4jri6Qu"
               "Ci0g6re4IOuwluyXkCDtnZTtlZwg7Iuk7IiY6rCAIOuRmCDsnojsirXri4jri6QuIHRlc3Rfc2l6ZSDrpbwg67m866m0IOqygOymnSDtj7Trk5zqsIAgMiwwMTYgw7cgNiA9IDMzNuyLnO"
               "qwhOydtCDrkJjqs6Ag7Y+J6regIFJNU0Ug64qUIDM3OS43MSDrjIDrnbwsIOyatOyYgeydmCDsmIjsuKEg6riw6rCEIDE2OOyLnOqwhOqzvCDri6Trpbgg7LGE7KCQ7J20IOuQqeuLiOuL"
               "pC4gMzIyLjUyIOuMgOulvCDqs4TsoIgg64KY7J2067iMIDE2OCDsnZggMzYzLjU1IOuMgOyZgCDrgpjrnoDtnogg7KCB7Ja0IO2ajOq3gOqwgCDrgqvri6Tqs6Ag7J2964qUIOqyg+uPhC"
               "Dsi6TsiJjsnoXri4jri6QuIOydtCDtmozqt4DripQg7LGE7KCQ7ZWY64qUIOyjvOydmCDsi6TsoJwg6riw7JiowrfqsJXsiJjrpbwg67Cb7JWY7Jy864uILCDquIjsmpTsnbwgMjPsi5zs"
               "l5AgMTY47Iuc6rCE7J2EIO2VnCDrsojsl5Ag64K064qUIOyYiOy4oeqzvCDqsJnsnYAg7KGw6rG07J20IOyVhOuLmeuLiOuLpC4KLSDsnbTrsogg7KO87JeQIO2VoCDsnbw6IOq1kOywqC"
               "DqsoDspp0g7L2U65Oc66W8IOyTuCDrlYwg7KCE7LKY66as7JmAIOuqqO2YleydhCBtYWtlX3BpcGVsaW5lIOycvOuhnCDrrLbslrQg67aE7ZWg66eI64ukIGZpdCDtlanri4jri6QuIOyd"
               "tOuvuCDsk7Qg64W47Yq467aB7JeQIFgg7KCE7LK066GcIGZpdF90cmFuc2Zvcm0g7ZWY64qUIOykhOydtCDsnojsnLzrqbQg7YyM7J207ZSE65287J24IOyViOycvOuhnCDsmK7quYHri4"
               "jri6QuIGFscGhhIOyymOufvCDsl6zrn6wg6rCS7J2EIOu5hOq1kO2VtCDqs6DrpbTripQg7ISk7KCV6rCS7J2ALCDqs6DrpbTripQg642wIOyTtCDqsoDspp0g7Y+065OcIO2Pieq3oOyd"
               "hCDqt7jrjIDroZwg67O06rOg7ZWY7KeAIOyViuqzoCDqs6DrpbTripQg642wIOyTsOyngCDslYrsnYAg6rWs6rCE7JeQ7IScIOuLpOyLnCDsuKHsoJXtlanri4jri6QuIOuztOqzoO2VoC"
               "DrlYzripQg6rCSIOyYhuyXkCDrtoTtlaDquLAoVGltZVNlcmllc1NwbGl0LCBuX3NwbGl0cz01LCB0ZXN0X3NpemU9MTY4KeyZgCDsoITsspjrpqzrpbwgZml0IO2VnCDqtazqsITsnYQg"
               "7KCB7Iq164uI64ukLgoK7L2U65Oc6rCAIOuLteydhCDrp4zrk5zripQg67Cp7IudLiDtjIzsnbTtlITrnbzsnbjsnYAgZml0KFhfY3ZbdHJdLCB5X2N2W3RyXSkg7J2EIOu2gOulvCDrlY"
               "wg67Cb7J2AIO2VmeyKtSDtj7Trk5wg7ZaJ7Jy866GcIOyKpOy8gOydvOufrOyZgCDrpr/sp4Ag7ZqM6reA66W8IOywqOuhgOuhnCBmaXQg7ZWp64uI64ukLiBwcmVkaWN0KFhfY3ZbdGVd"
               "KSDripQg6re4IO2Pieq3oOqzvCDtkZzspIDtjrjssKjroZwg6rKA7KadIO2PtOuTnOulvCDtkZzspIDtmZTrp4wg7ZWY6rOgIOuLpOyLnCBmaXQg7ZWY7KeAIOyViuyKteuLiOuLpC4g6r"
               "e4656Y7IScIOu2hO2VoOuniOuLpCDsiqTsvIDsnbzrn6zsnZggbWVhbl8g7J20IOuLpOultOqzoCwg67aE7ZWgIDEg7J2AIDnsm5QgMeydvO+9njEw7JuUIDE57J28IOqwkuunjCDslZXr"
               "i4jri6QuCg=="),
}
# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
               "eyJtMSI6eyLshYvsp7gg64KgIFNBUklNQSA17KO8IO2Pieq3oCBSTVNFIjpbIuybkOygkCA16rCc66eI64ukIO2VmeyKtSDqtazq"
               "sITsnYQgYGRmLmxvY1tGSVRfU1RBUlQ6b3JpZ2luLCAnZGVtYW5kJ11gIOuhnCDsnpjrnbwg64uk7IucIOygge2Vqe2VmOqzoCwg"
               "7JuQ7KCQIDHsi5zqsIQg65Kk67aA7YSwIDE2OOyLnOqwhOydhCDtlZwg67KI7JeQIOyYiOy4oe2VtCBSTVNFIOulvCDqtaztlanr"
               "i4jri6QuIO2VmeyKtSDsi5zsnpHsnYAgOeyblCAx7J2866GcIOuRoeuLiOuLpC4gMTHsm5QgMTbsnbwg7JuQ7KCQIO2VmOuCmOun"
               "jCDsnqzrqbQg7ZGcIDEg7J2YIOqwkuydtCDrgpjsmKTri4gg7JuQ7KCQIDXqsJzsnZgg7Y+J6reg7J247KeAIO2ZleyduO2VmOyE"
               "uOyalC4g6rCS7J2AIOOAjOydtOumhCA9IOyIq+yekCDrjIDjgI0g7ZWcIOykhOuhnCDstpzroKXtlanri4jri6QiLCLsnbQg7L2U"
               "65Oc6rCAIOybkOygkOuniOuLpCDslrTripAg6rWs6rCE7Jy866GcIO2VmeyKte2VmOqzoCDslrTripAgMTY47Iuc6rCE7J2EIOyx"
               "hOygkO2VmOuKlOqwgCJdLCLqs4TsoIgg64KY7J2067iMIDE2OCA17KO8IO2Pieq3oCBSTVNFIjpbIuqwmeydgCDsm5DsoJAgNeqw"
               "nMK36rCZ7J2AIDE2OOyLnOqwhOyXkOyEnCBgZGZbJ2RlbWFuZCddLnNoaWZ0KDE2OClgIOuhnCAx7KO87J28IOyghCDqsJnsnYAg"
               "7Iuc6rCBIOqwkuydhCDqurzrgrQgUk1TRSDrpbwg6rWs7ZWY6rOgIO2Pieq3oO2VqeuLiOuLpC4g6rOE7IiY66W8IOy2lOygle2V"
               "mOyngCDslYrripQg6rec7LmZ7J206528IOygge2Vqe2VmOyngCDslYrsirXri4jri6QiLCLsnbQg6rCS7J20IOyWtOuKkCAxNjjs"
               "i5zqsITsl5DshJwg66y07JeH7J2EIOyYiOy4oeycvOuhnCDsk7Qg6rKD7J246rCAIl0sInBsb3QiOlsi7ISg7J20IDPqsJzsnbjs"
               "p4Agwrcg6rCA66Gc7LaV7J20IOybkOygkCAxIOyXkOyEnCA1IOq5jOyngOyduOyngCDCtyDshKDrp4jri6Qg7KCQ7J20IDXqsJzs"
               "nbjsp4Agwrcg65GQIOy2leyXkCDrnbzrsqjsnYQg64us7JWY64qU7KeAIOywqOuhgOuhnCDrs7TshLjsmpQuIOqzhOygiCDrgpjs"
               "nbTruIwgMTY4IOuPhCDshKAg7ZWY64KY66GcIOq3uOumveuLiOuLpCIsIuuCtCDqt7jrprzsnZgg7Ja064qQIOy2lcK37Ja064qQ"
               "IOyEoOydtCDquLDspIDqs7wg7Ja06riL64Ks64qU7KeAIl0sImVycm9yIjpbIuyYpOulmCDrrLjsnqUg7LKrIOykhOydhCDqt7jr"
               "jIDroZwg7Yqc7YSw7JeQIOuztOyXrCDso7zqs6AsIOykgOu5hCDshYDsnbQg66eM65OgIOydtOumhChkZiDCtyBPUklHSU5TIMK3"
               "IEZJVF9TVEFSVCDCtyBPUkRFUlMgwrcgZml0X3NhcmltYXgp6rO8IHJtc2Ug64qUIOuLpOyLnCDrp4zrk6Tsp4Ag66eQ6rOgIOq3"
               "uOuMgOuhnCDsk7Drnbzqs6Ag67aE66qF7Z6IIOyggeycvOyEuOyalCIsIuydtCDsmKTrpZjqsIAg7Ja064qQIOykhOyXkOyEnCDs"
               "mZwg64Ks64qU7KeAIl19LCJtMiI6eyLrpr/sp4AgNeu2hO2VoCDtj4nqt6AgUk1TRSI6WyJgVGltZVNlcmllc1NwbGl0KG5fc3Bs"
               "aXRzPTUsIHRlc3Rfc2l6ZT0xNjgpYCDroZwg64KY64iE6rOgLCDrtoTtlaDrp4jri6QgYG1ha2VfcGlwZWxpbmUoU3RhbmRhcmRT"
               "Y2FsZXIoKSwgUmlkZ2UoYWxwaGE9MTAwKSlgIOydhCDtlZnsirUg7Y+065OcIO2WieycvOuhnOunjCBmaXQg7ZWp64uI64ukLiDq"
               "sJLsnYAg44CM7J2066aEID0g7Iir7J6QIOuMgOOAjSDtlZwg7KSE66GcIOy2nOugpe2VqeuLiOuLpCIsImB0ZXN0X3NpemU9MTY4"
               "YCDsnbQg67mg7KeA66m0IOqygOymnSDtj7Trk5zqsIAgMzM27Iuc6rCE7J20IOuQmOyWtCDqsJLsnbQg64us65287KeR64uI64uk"
               "LiDrqqjtmJXsnbQgYFJpZGdlKGFscGhhPTEwMClgIOydtOqzoCDshKTrqoUg67OA7IiY6rCAIGBYX2N2YCDsoITssrTsnbjsp4Dr"
               "j4Qg7ZmV7J247ZWY7IS47JqUIiwi7J20IOy9lOuTnOqwgCDqsoDspp0g7Y+065Oc66W8IOuqhyDsi5zqsITsnLzroZwg7J6Q66W0"
               "6rOgIOu2hO2VoOuniOuLpCDrrLTsl4fsnYQgZml0IO2VmOuKlOqwgCJdLCLrtoTtlaAgMSDsiqTsvIDsnbzrn6wg6riw7JioIO2P"
               "ieq3oCI6WyLrtoTtlaAgMSDtjIzsnbTtlITrnbzsnbjsnZggYG5hbWVkX3N0ZXBzWydzdGFuZGFyZHNjYWxlciddLm1lYW5fW1RF"
               "TVBfQ09MXWAg7J20IOq3uCDqsJLsnoXri4jri6QuIOyKpOy8gOydvOufrOulvCBgWF9jdmAg7KCE7LK066GcIO2VnCDrsoggZml0"
               "IO2WiOuLpOuptCAxMeyblOq5jOyngCDrk6TslrTqsIQg7Y+J6reg7J20IOuCmOyYteuLiOuLpC4g67aE7ZWg66eI64ukIO2VmeyK"
               "tSDtj7Trk5zroZwgZml0IO2WiOuKlOyngCDrs7TshLjsmpQiLCLrgrQg7Iqk7LyA7J2865+s6rCAIOyWtOuKkCDtlonsnLzroZwg"
               "Zml0IOuQmOyXiOuKlOqwgCJdLCJwbG90IjpbIuunieuMgOqwgCA16rCc7J247KeAIMK3IOqwgOuhnOy2leydtCDrtoTtlaAgMSDs"
               "l5DshJwgNSDquYzsp4Dsnbjsp4Agwrcg65GQIOy2leyXkCDrnbzrsqjsnYQg64us7JWY64qU7KeAIOywqOuhgOuhnCDrs7TshLjs"
               "mpQuIOunieuMgCDrhpLsnbTripQg67aE7ZWg66eI64ukIOq1rO2VnCDqsoDspp0g7Y+065OcIFJNU0Ug7J6F64uI64ukIiwi64K0"
               "IOq3uOumvOydmCDrp4nrjIDqsIAg66qHIOqwnOydtOqzoCDrrLTsl4fsnYQg7J6s6rOgIOyeiOuKlOyngCJdLCJlcnJvciI6WyLs"
               "mKTrpZgg66y47J6lIOyyqyDspITsnYQg6re464yA66GcIO2KnO2EsOyXkCDrs7Tsl6wg7KO86rOgLCDspIDruYQg7IWA7J20IOun"
               "jOuToCDsnbTrpoQoWF9jdiDCtyB5X2N2IMK3IFRFTVBfQ09MKeqzvCBybXNlIOuKlCDri6Tsi5wg66eM65Ok7KeAIOunkOqzoCDq"
               "t7jrjIDroZwg7JOw65286rOgIOu2hOuqhe2eiCDsoIHsnLzshLjsmpQiLCLsnbQg7Jik66WY6rCAIOyWtOuKkCDspITsl5DshJwg"
               "7JmcIOuCrOuKlOyngCJdfX0=")
_ASK = {"m1": "full", "m2": "blank"}   # 미션 2 는 튜터에 보낼 질문의 빈칸을 내가 채웁니다
_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 분명히 적습니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 틀 (문장으로 다듬는 것은 내가 합니다)",
             "self": "◼ 스스로 점검할 항목 틀 (이 미션은 힌트 없이 내가 명세합니다)"}
_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 줄 아래 코드를 보고 「{ask}」 에 3~5줄로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 줄 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 줄을 무엇으로 고칠지 한 줄로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : AI 코드 줄 아래 코드"}

_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set()}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 2단 힌트가 이 수를 봅니다
_orig_figure = getattr(plt.figure, "_orig", plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, **k):
    f = _orig_figure(*a, **k)
    _FIGS.append(f)
    return f


_figure._orig = _orig_figure
plt.figure = _figure                       # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in plt.get_fignums()]
    except Exception:
        return []


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다 (천 단위 쉼표 제거, 지수 표기 포함)"""
    picked = []
    for seg in re.findall(r"-?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?", text or ""):
        try:
            picked.append(float(seg.replace(",", "")))
        except ValueError:
            continue
    return picked


def _value_lines(txt, units):
    """값 채점이 볼 줄만 고릅니다 — 단위가 붙은 줄, 또는 「이름 = 값」 꼴의 줄.

    표를 통째로 띄우면 그 안에 정답이 우연히 섞여 있어, 「값을 골라 단위와 함께 출력한다」 는
    계약을 지키지 않고도 ✅ 가 열립니다. 그래서 단위를 달아 고른 줄이 하나라도 있으면 그
    줄들만 보고, 그런 줄이 아예 없을 때만 출력 전체를 봅니다.
    """
    picked = []
    for line in (txt or "").splitlines():
        if not line.strip():
            continue
        if "=" in line or any(unit and unit in line for unit in units):
            picked.append(line)
    return picked


def _label_lines(val_lines, label):
    """라벨이 든 값줄만 골라, 그 줄에서 라벨 뒤(「=」·「:」 뒤) 토막을 돌려줍니다.

    값줄의 숫자를 한 주머니에 담아 띠와 대조하면 **옆 칸의 값**이 이 칸을 헛통과시킵니다 —
    부호 칸 둘이 같은 띠일 때 한쪽에 찍힌 1 이 다른 칸까지 열어 주던 자리입니다. 그래서
    라벨이 든 줄이 있으면 그 줄의 라벨 뒤만 봅니다(없으면 부르는 쪽이 값줄 전체로 물러납니다).
    라벨 찾기는 정확히 같은 문자열 → 못 찾으면 공백·괄호를 걷어낸 꼴, 두 단입니다.
    """
    def _tighten(txt):                     # 공백과 괄호(안의 말까지) 를 걷어낸 글자 · 원래 자리
        keep, depth = [], 0
        for i, ch in enumerate(txt or ""):
            if ch in "([":
                depth += 1
            elif ch in ")]":
                depth = max(0, depth - 1)
            elif depth == 0 and not ch.isspace():
                keep.append(i)
        return "".join((txt or "")[i] for i in keep), keep

    tight_label = _tighten(label)[0]
    tails = []
    for line in val_lines:
        end = line.find(label) + len(label) if label and label in line else -1
        if end < 0 and tight_label:                       # 2단 — 공백·괄호를 걷어내고 다시 봅니다
            tight_line, keep = _tighten(line)
            at = tight_line.find(tight_label)
            end = (keep[at + len(tight_label) - 1] + 1) if at >= 0 else -1
        if end < 0:
            continue
        tail = line[end:]
        cut_at = [p for p in (tail.find("="), tail.find(":")) if p >= 0]
        tails.append(tail[min(cut_at) + 1:] if cut_at else tail)
    return tails


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _seg_lines(ax):
    """`vlines` · `hlines` 로 그은 선 — Line2D 가 아니라 **LineCollection** 이라 `ax.lines` 에 없습니다.

    토막 하나를 `(가로값, 세로값)` 한 쌍으로 펴서 돌려줍니다. 이 길을 열어 두지 않으면
    `plt.vlines(...)` 로 화면에 분명히 그어 놓은 세로선이 「세로선이 아예 없다」 로 판정돼,
    학생이 고칠 곳 없는 결함을 고치러 가게 됩니다.
    """
    picked = []
    for c in getattr(ax, "collections", []):
        get_segs = getattr(c, "get_segments", None)      # LineCollection 에만 있는 메서드입니다
        if get_segs is None:                              # 산점도(PathCollection)·띠(PolyCollection)는 선이 아닙니다
            continue
        try:
            segs = get_segs()
        except Exception:
            continue
        for seg in segs:
            xy = np.asarray(seg, dtype=float).reshape(-1, 2)
            if xy.shape[0] >= 2:
                picked.append((xy[:, 0], xy[:, 1]))
    return picked


def _lines(ax, min_pts=10):
    """축의 선을 곡선(점 `최소점` 개 이상)과 수직선(x 가 같은 두 점)으로 갈라 돌려줍니다.

    문턱을 인자로 연 것은 오늘 그림의 선이 **점 일곱 개**(검증 7일)이기 때문입니다. 문턱을
    10 으로 고정해 두면 규격대로 그린 학생의 선이 「곡선이 0개다」 로 읽혀 고칠 곳 없는
    결함을 고치러 가게 됩니다.
    """
    cur, vline = [], []
    cands = []
    for ln in ax.lines:                     # 선 하나가 깨져도 나머지는 살린다 — `axvline(x=날짜)` 처럼 x 가
        try:                                #  Timestamp 인 세로선(add_changepoints_to_plot)은 날짜를 숫자로 바꿔 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        cands.append((xd, yd))
    cands += _seg_lines(ax)                     # `plt.vlines` 로 그은 수직선도 여기서 합류합니다
    for xd, yd in cands:
        if xd.size >= min_pts:
            cur.append((xd, yd))
        elif xd.size == 2 and np.isclose(xd[0], xd[1]):
            vline.append(float(xd[0]))
    return cur, vline


def _marker_lines(ax):
    """마커를 찍은 Line2D — `plot(x, y, "o")` 든 `plot(x, y, marker="o")` 든 점으로 셉니다.

    선으로 이으면서 점도 함께 찍은 선(`linestyle` 이 기본 실선)까지 세는 것은, 점추정을 그렇게
    찍은 그림이 「점추정 마커가 0개다」 로 ❌ 나던 자리이기 때문입니다. 점 계수는 linestyle 과
    무관하고, 마커를 달지 않은 선만 점에서 뺍니다.
    """
    picked = []
    for ln in ax.lines:
        marker = str(ln.get_marker()).strip()
        if marker in ("", "None", "none"):     # 마커 없는 선은 점이 아닙니다 — 선 모양은 보지 않습니다
            continue
        if marker in ("|", "_"):               # errorbar 의 capsize 캡은 점이 아닙니다 — 구간 끝을 그은 짧은 선입니다
            continue
        try:                                # 날짜 x 인 선 하나가 나머지 점까지 죽이지 않게 갈라 받는다
            xd = np.asarray(ln.get_xdata(), dtype=float).reshape(-1)
            yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
        except (TypeError, ValueError):
            try:
                import matplotlib.dates as _mdates
                xd = np.asarray(_mdates.date2num(ln.get_xdata()), dtype=float).reshape(-1)
                yd = np.asarray(ln.get_ydata(), dtype=float).reshape(-1)
            except Exception:
                continue
        if xd.size and xd.size == yd.size:
            picked.append((ln, xd, yd))
    return picked


def _diagonal(ax, skip=(), xy=None):
    """y = x 인 기준선이 점이 놓인 범위를 가로지르는가 — 선이 있기만 하면 통과가 아닙니다"""
    drop_ids = {id(ln) for ln in skip}
    cands = [(np.asarray(ln.get_xdata(), dtype=float).reshape(-1),
           np.asarray(ln.get_ydata(), dtype=float).reshape(-1))
          for ln in ax.lines if id(ln) not in drop_ids]
    cands += _seg_lines(ax)                     # LineCollection 으로 그은 y = x 기준선도 셉니다
    for xd, yd in cands:
        if xd.size < 2 or xd.size != yd.size:
            continue
        if not (np.isfinite(xd).all() and np.isfinite(yd).all()):
            continue
        width = float(xd.max() - xd.min())
        if width <= 0:                                    # 수직선은 대각선이 아닙니다
            continue
        if not np.allclose(yd, xd, rtol=1e-3, atol=0.02 * width):
            continue                                   # 세로가 가로를 따라가지 않으면 넘어갑니다
        if xy is not None and getattr(xy, "size", 0):
            pt_span = float(xy[:, 0].max() - xy[:, 0].min())
            overlap = float(min(xd.max(), xy[:, 0].max()) - max(xd.min(), xy[:, 0].min()))
            if pt_span > 0 and overlap < 0.4 * pt_span:          # 점이 놓인 구간을 거의 덮지 못하는 토막 선
                continue
        return True
    return False


def _near(val, ref, factor=1.5):
    """실측값이 기준값의 배수 안에 드는가 — 눈금·표본 차이는 넘어갑니다"""
    try:
        val = float(val)
    except (TypeError, ValueError):
        return False
    if ref == 0:
        return abs(val) < 1e-9
    return val * ref > 0 and abs(ref) / factor <= abs(val) <= abs(ref) * factor


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과)"""
    gaps = []
    if spec.get("kind") == "line":
        cur, vline = _lines(ax, spec.get("min_points", 10))
        want = spec.get("n_series", len(cur))
        if len(cur) != want:
            gaps.append(f"선이 {len(cur)}개다 (기준 {want}개)")
        if cur and "n_points" in spec:
            n_spec = spec["n_points"]                   # 정수면 딱 그 개수, 두 값이면 그 사이
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            off_range = [int(x.size) for x, _ in cur if not (n_lo <= x.size <= n_hi)]
            if off_range:
                ref_text = f"{n_lo}개" if n_lo == n_hi else f"{n_lo}개 이상 {n_hi}개 이하"
                gaps.append(f"점이 {off_range[0]}개인 선이 있다 (기준 {ref_text})")
        if cur:
            xd, yd = max(cur, key=lambda t: t[0].size)
            lo, hi = spec.get("x_range", (xd.min(), xd.max()))
            slack = 0.2 * (hi - lo)
            if not (xd.min() <= lo + slack and xd.max() >= hi - slack):
                gaps.append(f"가로축이 {xd.min():.3g}~{xd.max():.3g} 만 포함한다 (기준 {lo:g}~{hi:g})")
            if "peak_x" in spec:
                p0, p1 = spec["peak_x"]
                peak = float(xd[yd.argmax()])         # 로그우도면 봉우리가 그 자리에 섭니다
                trough = float(xd[yd.argmin()])         # NLL 로 그렸으면 골짜기가 그 자리입니다
                if not (p0 <= peak <= p1 or p0 <= trough <= p1):
                    gaps.append(f"곡선의 최댓값 위치가 {peak:.4f}, 최솟값 위치가 {trough:.4f} 다"
                                f" (기준 {p0}~{p1} — 둘 중 하나가 이 구간 안에 있어야 합니다)")
        if cur and ("봉우리_x" in spec or "골짜기_x" in spec):
            # 부호가 뜻을 가진 곡선(충격반응) — 위로 솟는 곡선과 아래로 꺼지는 곡선을 따로 봅니다.
            # 반응을 통째로 뒤집어 그리면 봉우리·골짜기의 자리는 그대로라 자리만 봐서는 걸러지지 않습니다
            if "봉우리_x" in spec:
                a0, a1 = spec["봉우리_x"]
                if not any(float(np.nanmax(y)) > 0 and a0 <= float(x[int(np.nanargmax(y))]) <= a1
                           for x, y in cur):
                    gaps.append(f"위로 솟아 {a0:g}~{a1:g} 시간 구간에서 최댓값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
            if "골짜기_x" in spec:
                b0, b1 = spec["골짜기_x"]
                if not any(float(np.nanmin(y)) < 0 and b0 <= float(x[int(np.nanargmin(y))]) <= b1
                           for x, y in cur):
                    gaps.append(f"아래로 꺼져 {b0:g}~{b1:g} 시간 구간에서 최솟값에 이르는 곡선이 없다"
                                " (부호가 뒤집혔으면 반응을 반대 방향으로 읽은 것입니다)")
        if "vline_x" in spec:
            v0, v1 = spec["vline_x"]
            if not any(v0 <= v <= v1 for v in vline):
                gaps.append("최댓값 위치를 가리키는 세로선이 기준 구간에 없다"
                            + (f" (지금 {vline[0]:.4f})" if vline else " (세로선이 아예 없다)"))
    elif spec.get("kind") == "bar":
        heights = [float(p.get_height()) for p in ax.patches]
        if "n" in spec:
            n_spec = spec["n"]                          # 정수면 딱 그 개수, 두 값이면 그 사이(예: 시차 24~400)
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            ref_text = f"{n_lo}개" if n_lo == n_hi else f"{n_lo}개 이상 {n_hi}개 이하"
            if not (n_lo <= len(heights) <= n_hi):
                gaps.append(f"막대가 {len(heights)}개다 (기준 {ref_text})")
        elif not heights:                              # 막대 수 기준이 없는 경우에도 빈 축은 통과시키지 않습니다
            gaps.append("막대가 하나도 없다 (이 축에서 막대를 찾지 못했습니다)")
        if heights and "ymax" in spec and not (spec["ymax"][0] <= max(heights) <= spec["ymax"][1]):
            gaps.append(f"가장 높은 막대가 {max(heights):.4f} 다 (기준 {spec['ymax']})")
    elif spec.get("kind") == "scatter":
        markers = _marker_lines(ax)                              # plot(x, y, "o") 로 찍은 점도 셉니다
        pt_sets = [np.asarray(c.get_offsets(), dtype=float).reshape(-1, 2)
              for c in ax.collections if not hasattr(c, "get_segments")]   # 선 묶음은 점으로 세지 않습니다
        pt_sets += [np.column_stack([xd, yd]) for _, xd, yd in markers]
        n_pts = int(sum(p.shape[0] for p in pt_sets))
        if "n_points" in spec:
            n_spec = spec["n_points"]
            n_lo, n_hi = n_spec if isinstance(n_spec, tuple) else (n_spec, n_spec)
            if not (n_lo <= n_pts <= n_hi):
                gaps.append(f"점이 {n_pts}개다 (기준 {n_lo}개 — 가로축 눈금 하나에 점 하나,"
                            f" 강조로 덧찍는 점은 {n_hi - n_lo}개까지 넘어갑니다)")
        xy = np.vstack([p for p in pt_sets if p.size]) if any(p.size for p in pt_sets) else None
        if xy is not None:
            xy = xy[np.isfinite(xy).all(axis=1)]
        if xy is not None and xy.size:
            xs, ys = xy[:, 0], xy[:, 1]
            if "x_range" in spec:
                lo, hi = spec["x_range"]
                if not (_near(xs.min(), lo) and _near(xs.max(), hi)):
                    gaps.append(f"가로축 값이 {xs.min():.3g}~{xs.max():.3g} 다 (기준 {lo:g}~{hi:g})")
            if "y_range" in spec:
                lo, hi = spec["y_range"]
                if not (_near(ys.min(), lo) and _near(ys.max(), hi)):
                    gaps.append(f"세로축 값이 {ys.min():.4g}~{ys.max():.4g} 다 (기준 {lo:g}~{hi:g})")
            if "ymax" in spec and not (spec["ymax"][0] <= ys.max() <= spec["ymax"][1]):
                gaps.append(f"세로축 최댓값이 {ys.max():.4g} 다"
                            f" (기준 {spec['ymax'][0]:g}~{spec['ymax'][1]:g})")
            if "gap_min" in spec:
                spread = float(np.abs(ys - xs).max())
                if spread < spec["gap_min"]:
                    gaps.append(f"두 축의 값이 가장 크게 벌어진 곳도 {spread:.4f} 뿐이다"
                                f" (기준 {spec['gap_min']:g} 이상)")
        if spec.get("xscale") and ax.get_xscale() != spec["xscale"]:
            gaps.append(f"가로축이 {ax.get_xscale()} 눈금이다 (기준 {spec['xscale']} 눈금)")
        if spec.get("diag") and not _diagonal(ax, skip=[ln for ln, _, _ in markers], xy=xy):
            gaps.append("두 축의 값이 같은 점을 잇는 대각선(y = x)이 점이 놓인 구간에 없다")
    if spec.get("축라벨") and not (ax.get_xlabel().strip() and ax.get_ylabel().strip()):
        gaps.append("가로축·세로축 라벨 중 빈 것이 있다")
    return gaps


def _ref_plot_m1():
    """미션 1 의 참조 그림 — 원점 5개에서 세 모형이 받은 RMSE(채점 기준에 적어 둔 값)로 세 선을 그립니다"""
    ref = _ANS.get("m1", {}).get("ref")
    if not ref:
        return None
    xs = list(range(1, 6))
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    for name, color in [("둘째 날 ARIMA", COL["연회색"]), ("셋째 날 SARIMA", COL["하늘색"]),
                        ("계절 나이브 168", COL["남색"])]:
        a.plot(xs, list(ref[name]), marker="o", color=color, lw=1.8, label=name)
    a.set_xticks(xs)
    a.set_xlabel("예측 원점")
    a.set_ylabel("그 주의 RMSE (대)")
    a.legend()
    return f


def _ref_plot_m2():
    """미션 2 의 참조 그림 — 분할 5개의 검증 폴드 RMSE(채점 기준에 적어 둔 값)로 막대 5개를 그립니다"""
    ref = _ANS.get("m2", {}).get("ref")
    if not ref:
        return None
    heights = list(ref["분할별"])
    f = _orig_figure(figsize=(FIG_W * 0.8, 3.4))
    a = f.add_subplot(111)
    a.bar(range(1, 6), heights, width=0.6, color=COL["하늘색"])
    a.set_ylim(0, max(heights) * 1.18)          # 막대 위 숫자가 잘리지 않게 위를 조금 띄운다
    for x, y in zip(range(1, 6), heights):
        a.text(x, y + max(heights) * 0.03, f"{y:.2f}", ha="center", color=COL["회색"])
    a.set_xticks(range(1, 6))
    a.set_xlabel("분할")
    a.set_ylabel("검증 폴드 RMSE (대)")
    return f


_REF = {"m1": _ref_plot_m1, "m2": _ref_plot_m2}   # 미션마다 참조 그림을 그리는 함수 하나


def _show(fig):
    """채점 훅 안에서 그림을 화면에 올립니다 (인라인 백엔드는 이 시점에 plt.show() 가 듣지 않습니다)"""
    try:
        from IPython.display import display
        display(fig)
    except Exception:
        plt.show()


def _ref_show(fig, key):
    """내 그림과 참조 그림을 나란히 — 내 그림이 없으면 참조 그림도 띄우지 않습니다"""
    if not _axes(fig) or key not in _REF:
        return
    ref = None
    try:
        ref = _REF[key]()
        if ref is None:
            return
        panels = [(fig, "내 그림"), (ref, "참조 그림")]
        f = _orig_figure(figsize=(FIG_W, 3.4))
        for i, (g, title) in enumerate(panels):
            a = f.add_subplot(1, len(panels), i + 1)
            buf = io.BytesIO()
            (g if hasattr(g, "savefig") else g.figure).savefig(buf, format="png", dpi=110,
                                                               bbox_inches="tight")
            buf.seek(0)
            a.imshow(plt.imread(buf))
            a.axis("off")
            a.set_title(title, fontsize=11)
        _show(f)
        plt.close(f)
    except Exception:
        print("ⓘ 참조 그림을 나란히 놓지 못했습니다 — 위 채점 줄은 그대로 유효합니다")
    finally:
        if ref is not None:
            plt.close(ref)


def _one_line(e, n=120):
    """예외를 한 줄로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다"""
    txt = str(e).replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt


def _prompt1(n=70):
    """지금 셀의 지시 첫 줄 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in str(_get("PROMPT") or "").split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                    "여기에 내 말로 적습니다")


def _strip(txt):
    """빈칸·줄표·가운뎃점을 떼어 낸 알맹이 — 자리표시자 대조는 이 꼴로 합니다"""
    return re.sub(r"[\s—·:,\-]+", "", str(txt or ""))


_PLACEHOLDERS_STRIPPED = {_strip(t) for t in _PROMPT_PLACEHOLDERS}


def _prompt_missing():
    """지시 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    try:
        txt = str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "").strip()
    except Exception:
        txt = str(_get("PROMPT") or "").strip()
    stripped = _strip(txt)                          # 길이보다 먼저 본다 — 견본을 그대로 둔 자리가 있다
    if not stripped:
        return "PROMPT 입력란이 비어 있습니다"
    if stripped in _PLACEHOLDERS_STRIPPED:
        return "PROMPT 입력란이 자리표시자·견본 그대로입니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 줄부터 채점 표를 닫는 `─` 구분선 줄까지를
    출력 패널이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


def judge(key, text="", figs=(), error=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다"""
    spec = _ANS.get(key)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    reason = _prompt_missing()                        # 값·그래프보다 먼저 — 내 말로 시킨 자리가 있는가
    _locked = bool(reason)
    if reason:
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print("─" * 54)
        return False
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        print(f"❌ 실행 오류 — {type(error).__name__}: {_one_line(error)}")
        hint, question = _HINT.get(key, {}).get("error", ("", "이 오류가 어느 줄에서 왜 났는지"))
        if hint:
            print(f"   힌트 : {hint}")
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        print("  " + _ASK_FORM[form].format(name="실행 오류", prompt=_prompt1(), ask=question))
        print("─" * 54)
        return False
    txt = text or ""
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    units = [str(ref[2]) for ref in spec.get("values", {}).values() if len(ref) > 2]
    val_lines = _value_lines(txt, units)                    # 단위·「=」 가 붙은 줄이 있으면 그 줄만 봅니다
    nums = _nums("\n".join(val_lines)) if val_lines else _nums(txt)
    table_flood = not val_lines and len(nums) > 30        # 골라낸 자리 없이 숫자만 잔뜩 = 표를 통째로 띄운 모양
    if table_flood:
        print(f"❌ 값 — 단위도 「이름 = 값」 도 없이 숫자가 {len(nums)}개 나왔습니다 — 표를"
              " 통째로 띄우면 어느 숫자가 답인지 채점이 고르지 못합니다."
              " 값은 「이름 = 숫자 단위」 한 줄로 출력하세요")
        nums = []
    for label, ref in spec.get("values", {}).items():
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        tail = _label_lines(val_lines, label) if nums else []   # 이 라벨이 든 행의 「=」 뒤만 — 옆 칸과 안 섞이게
        fell_back = bool(nums) and not tail             # 라벨이 든 행이 없을 때만 값줄 전체로 물러납니다
        hits = [v for v in (_nums("\n".join(tail)) if tail else nums) if lo <= v <= hi]
        if hits:
            print(f"✅ {label} — 출력에서 {hits[0]:,.6g} 를 찾았습니다")
            if unit and unit not in txt:
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif table_flood:
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 「이름 = 숫자 {unit}」 한 줄로 골라 출력하면 채점이 읽습니다")
        else:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 결정 16 — 첫 ❌ 에는 기준 범위를 감춘다
            print(f"❌ {label} — 기준에 닿는 숫자를 출력에서 찾지 못했습니다"
                  + (f" (기준 범위 {lo:g} ~ {hi:g})" if _prev_fail else ""))
            if fell_back:                            # 라벨 행을 못 찾아 값줄 전체를 본 자리
                print(f"   ⓘ 라벨 「{label}」 이 든 행을 찾지 못해 값줄 전체에서 찾았습니다"
                      " — 「이름 = 숫자 단위」 한 행으로 출력하면 그 행만 보고 채점합니다")
    for phrase in spec.get("forbid", ()):            # 함정 미션 — 전임자의 판단 줄·옛 이름이 남으면 통과시키지 않습니다
        if phrase in txt:
            passed = False
            if ("남은 줄", "forbid") not in failed:
                failed.append(("남은 줄", "forbid"))
            print(f"❌ 남은 줄 — 출력에 「{phrase}」 문구가 남아 있습니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다)")
        else:
            need_axes = int(plot_spec.get("n_axes", 1))   # 그림 두 장을 나란히 요구하는 기준이면 2 입니다
            ok_axes, gaps = 0, None
            for ax in axs:                 # 기준에 맞는 축이 필요한 수만큼 있으면 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림에서 데이터를 뽑다 걸렸습니다 ({type(e).__name__})"]
                if not this_gaps:
                    ok_axes += 1
                    if ok_axes >= need_axes:
                        break
                elif gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            if ok_axes < need_axes:
                passed = False
                failed.append(("그래프", "plot"))
                extra_msgs = [f"기준에 맞는 그림이 {ok_axes}개다 (기준 {need_axes}개)"] if need_axes > 1 else []
                print("❌ 그래프 — " + " · ".join(extra_msgs + (gaps or [])))
            else:
                print("✅ 그래프 — 기준대로 그렸습니다")
            _ref_show(figs[0] if figs else None, key)
    tbl = _HINT.get(key, {})
    attempt = _TRY[key] = _TRY.get(key, 0) + (1 if failed else 0)
    for name, hkey in failed:
        tiers = tbl.get(hkey, ("",))
        tier = tiers[:-1] if len(tiers) > 2 else tiers[:1]   # 2단 힌트는 마지막 원소가 튜터에 물을 것입니다
        hint = tier[min(attempt, len(tier)) - 1] if tier else ""
        if hint:
            print(f"   힌트({name}) : {hint}")
    if failed:
        form = _ASK.get(key, "full")
        print(_ASK_HEAD[form])
        for name, hkey in failed:
            question = (tbl.get(hkey, ("", ""))[-1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                or "이 항목이 왜 ❌ 로 남는지"
            print("  " + _ASK_FORM[form].format(name=name, prompt=_prompt1(), ask=question))
    elif _locked:                                  # 값·그림은 닿았지만 내 말로 시킨 자리가 아직 비었습니다
        print("ⓘ 값과 그림은 기준에 닿았습니다 — PROMPT 에 고칠 지시를 내 말로 쓰고"
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 견줘 보세요")
        _reveal_show(base64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked


class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """첫 줄이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다"""
    head_line = (src or "").split("\n")[0]
    m = re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
    if not m or "프롬프트" not in head_line:                           # 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다
        return None
    return "m" + m.group(2) if m.group(2) else "과제"


def _pre(info=None):
    """「PROMPT 셀」면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        tag = _tag(getattr(info, "raw_cell", ""))
        _STATE["tag"] = tag
        if tag is None or tag not in _ANS:
            return
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["buf"], _STATE["out"] = io.StringIO(), sys.stdout
        sys.stdout = _Tee(sys.stdout, _STATE["buf"])
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다"""
    tag, buf, out = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out")
    _STATE["tag"], _STATE["buf"], _STATE["out"] = None, None, None
    if out is not None:
        sys.stdout = out
    if tag is None or tag not in _ANS:
        return
    seen = {id(f) for f in _FIGS}
    figs_all = list(_FIGS) + [f for f in _open_figs()
                         if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
    txt = buf.getvalue() if buf is not None else ""
    last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
    if last_val is not None:
        try:
            txt = txt + "\n" + str(last_val)
        except Exception:
            pass
    try:
        judge(tag, txt, figs_all, error=getattr(result, "error_in_exec", None))
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")


try:
    _ip = get_ipython()
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _fn)
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 서울 공유자전거 준비

자료는 **서울 공유자전거 시간당 대여 기록**(UCI 머신러닝 저장소)입니다. 2017년 12월 1일부터 2018년 11월 30일까지 8,760행입니다. 휴무 295시간을 1주일 전 같은 시각 값으로 채운 보정 수요 `demand` 를 씁니다. 학습 구간은 9월 1일～11월 16일 1,848시간이고, 검증 구간은 11월 17～23일 168시간입니다.

후보 가운데 계절 나이브 24 는 11월 16일 하루의 값 24개를 7번 되풀이한 예측입니다. 「오늘의 준비」 셀은 30초 안팎 걸리고 아래 셋을 출력합니다.

- 표 1 · 11월 16일 예측 원점 한 주의 RMSE (대): 후보 4개를 같은 원점·같은 168시간에서 채점한 값과 AIC 차이
- 표 2 · 같은 5주를 채점한 선형회귀의 평균 RMSE (대): 같은 모형을 분할기만 바꿔 채점한 값
- 그림 1 · 9월 1일～11월 23일 주 평균 수요 (대/시간)

In [ ]:
# 오늘의 준비 — 읽고 실행만 합니다
# 서울 공유자전거로 표 1 · 표 2 · 그림 1 을 만듭니다. 셋째 날 SARIMA 적합이 있어 30초 안팎 걸립니다
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, TimeSeriesSplit

URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")        # 8,760행 x 14열
#  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일창에 올린 뒤
#  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",      # 위치 기반 리네임
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")    # 01/12/2017 = 2017년 12월 1일
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"                       # 등간격 1시간
_s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
_y = _s.fillna(_s.shift(24 * 7))          # 1주일 전 같은 시각 값으로 채운다
df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로

# 표 1: 11월 16일 23시 예측 원점에서 168시간을 한 번에 예측한 후보 4개
train = df.loc["2018-09-01 00:00":"2018-11-16 23:00", "demand"]      # 학습 1,848시간
valid = df.loc["2018-11-17 00:00":"2018-11-23 23:00", "demand"]      # 검증 168시간
fits = {name: SARIMAX(train, order=(1, 0, 1), seasonal_order=so).fit(disp=False)
        for name, so in [("둘째 날 ARIMA", (0, 1, 0, 24)), ("셋째 날 SARIMA", (1, 1, 1, 24))]}
week_rmse = {"계절 나이브 168": rmse(valid, df["demand"].shift(168).loc[valid.index]),   # 1주일 전 같은 시각 값
             "계절 나이브 24": rmse(valid, np.tile(train.values[-24:], 7))}           # 11월 16일 하루를 7번
week_rmse.update({name: rmse(valid, res.forecast(168)) for name, res in fits.items()})
aic_gap = fits["둘째 날 ARIMA"].aic - fits["셋째 날 SARIMA"].aic   # 학습 구간의 1시간 뒤 예측으로 계산한 차이
print("표 1 · 11월 16일 예측 원점 한 주의 RMSE (대)")
for name, v in week_rmse.items():
    print(f"  {name:<12} {v:8.2f}")
print(f"  학습 구간 AIC 는 셋째 날 SARIMA 가 {aic_gap:,.2f} 낮다")

# 표 2: 같은 선형회귀를 분할기 두 개로 채점하고, 둘 다 마지막 5주(10월 20일～11월 23일)의 RMSE 를 평균
d_cv = df.loc["2018-09-01 00:00":"2018-11-23 23:00"]                 # 2,016시간
X_cv = pd.get_dummies(pd.DataFrame({"hour": d_cv.index.hour.astype(str),
                                    "dow": d_cv.index.dayofweek.astype(str)}, index=d_cv.index))
X_cv[["temp", "rain"]] = d_cv[["temp", "rain"]].values               # 시각 더미 24 · 요일 더미 7 · 기온 · 강수
X_cv, y_cv = X_cv.astype(float).values, d_cv["demand"].values
oof, n_later = np.zeros(len(y_cv)), np.zeros(len(y_cv))
for tr, te in KFold(5, shuffle=True, random_state=SEED).split(X_cv):   # 행 순서를 섞어 5개 폴드로
    oof[te] = LinearRegression().fit(X_cv[tr], y_cv[tr]).predict(X_cv[te])
    n_later[te] = [np.sum(tr > t) for t in te]                        # 채점 시각보다 뒤인데 학습에 들어간 시각 수
weeks = list(TimeSeriesSplit(n_splits=5, test_size=168).split(X_cv))  # 검증 폴드 = 마지막 5주
shuf_rmse = [rmse(y_cv[te], oof[te]) for _, te in weeks]
tss_rmse = [rmse(y_cv[te], LinearRegression().fit(X_cv[tr], y_cv[tr]).predict(X_cv[te])) for tr, te in weeks]
shuf_later = float(np.mean([n_later[te].mean() for _, te in weeks]))
tss_later = [int(np.sum(tr > te.min())) for tr, te in weeks]
print("표 2 · 같은 5주를 채점한 선형회귀의 평균 RMSE (대)")
print(f"  KFold(shuffle=True)              평균 RMSE {np.mean(shuf_rmse):7.2f}"
      f"   채점 시각보다 뒤인 학습 시각 평균 {shuf_later:4.0f}시간")
print(f"  TimeSeriesSplit(test_size=168)   평균 RMSE {np.mean(tss_rmse):7.2f}"
      f"   채점 시각보다 뒤인 학습 시각 평균 {np.mean(tss_later):4.0f}시간")

# 그림 1: 토요일～금요일 한 주씩 평균한 수요 12주, 회색 배경이 표 2 가 채점하는 5주
weekly = d_cv["demand"].resample("7D").mean()                         # 9월 1일(토)부터 7일씩
fig, ax = plt.subplots(figsize=(FIG_W, 3.4))
ax.plot(weekly.index, weekly.values, marker="o", color=COL["파랑"], lw=1.8)
ax.axvspan(pd.Timestamp("2018-10-18 12:00"), pd.Timestamp("2018-11-18 12:00"), color=COL["연회색"], alpha=0.35)
ax.text(pd.Timestamp("2018-10-20"), 1270, "표 2 가 채점하는 5주", color=COL["회색"])
ax.set_ylim(0, 1400)
ax.set_title("그림 1 · 9월 1일～11월 23일 주 평균 수요 (대/시간)")
ax.set_ylabel("주 평균 수요 (대/시간)")
datefmt(ax, n=7)
plt.show()

## 예측 원점과 학습 폴드 확인

> **[문제] 한 주 결과와 원점 5개 평균이 엇갈리면 무엇으로 정하나**
>
> 예를 들어 모형 A 가 11월 16일 원점 한 주에서는 계절 나이브 168 보다 50대 크게 빗나갔고, 예측 원점 5개 평균에서는 20대 작게 빗나갔다고 합시다.
> 
> **핵심 키워드:** 원점 5개 평균이 기준, 한 주의 순위는 원점마다 달라진다, 평가표에 조건을 함께 적는다
>
> 1. 가장 최근 주가 운영과 가까우니 한 주 결과를 따라 투입하지 않는다
> 2. 원점 5개 평균이 베이스라인보다 작으니 투입을 논의하고, 원점 수와 학습 구간을 함께 적는다
> 3. 두 결과가 엇갈리니 한 주 차이와 5개 평균 차이를 더해 부호로 정한다

> **[문제] 교차 검증에서 스케일러는 어느 자료로 fit 하나**
>
> **핵심 키워드:** 분할마다 학습 폴드로 fit, 요약 값도 누수가 된다, make_pipeline
>
> 1. 분할마다 그 분할의 학습 폴드로 fit 한다
> 2. 전체 자료로 한 번 fit 해 모든 분할에 쓴다
> 3. 분할마다 학습 폴드와 검증 폴드를 합쳐 fit 한다

## 서울 공유자전거 채점표로 판단하기

아래 두 문항은 「오늘의 준비」 셀 출력을 읽고 답합니다. 표 1 은 11월 16일 원점 한 주의 채점표입니다. 표 2 는 같은 5주를 분할기만 바꿔 두 번 채점한 값입니다.

<img src="https://otexts.com/fpp3/fpp_files/figure-html/cv1-1.png" alt="행마다 학습 시점(파랑)이 하나씩 늘고 바로 뒤 시점 하나(주황)를 채점하는 시계열 교차 검증 그림" width="480"/>
<p align="center"><em>▲ 예측 원점을 한 시점씩 옮기며 같은 방식으로 다시 채점하는 시계열 교차 검증. 파랑은 학습 시점, 주황은 채점 시점, 회색은 그 행에서 쓰지 않는 시점이다. 어느 행에서도 채점 시점보다 뒤의 값은 학습에 들어가지 않는다 (출처: Hyndman &amp; Athanasopoulos, Forecasting: Principles and Practice 3rd ed., §5.10)</em></p>

> **[문제] 한 주 채점표로 내릴 수 있는 판단**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 1 · 11월 16일 예측 원점 한 주의 RMSE (대)」를 봅니다. 이 표로 내릴 수 있는 판단은 무엇일까요?
> 
> **핵심 키워드:** 베이스라인과 같은 주에서 나란히, AIC 는 1시간 뒤 오차, 한 주로 순위를 정하지 않는다
>
> 1. AIC 가 더 낮은 셋째 날 SARIMA 를 운영에 투입한다
> 2. 이 주에 덜 빗나간 둘째 날 ARIMA 를 운영에 투입한다
> 3. 두 모형 모두 투입하지 않고, 원점을 옮겨 여러 주로 다시 채점한다

> **[문제] 더 작게 나온 교차 검증 RMSE 를 믿어도 되나**
>
> 「오늘의 준비」 셀을 실행하면 나오는 「표 2 · 같은 5주를 채점한 선형회귀의 평균 RMSE (대)」를 봅니다. 두 분할기 가운데 평균 RMSE 가 더 작은 쪽을 교차 검증 결과로 보고해도 될까요?
> 
> **핵심 키워드:** 채점 시각보다 뒤의 시각이 학습에 들어간 값, 운영에서 할 수 없는 채점, 분할기와 검증 폴드 길이를 함께 보고
>
> 1. 된다. 같은 5주를 채점했으니 더 작은 값을 받은 분할기가 더 좋은 평가 방식이다
> 2. 안 된다. 채점 시각보다 뒤의 시각을 학습에 포함한 값이라, 금요일 23시에는 할 수 없는 채점이다
> 3. 된다. 시드를 고정해 다시 실행해도 같은 값이 나오니 믿을 수 있는 값이다

## 미션 1 — 예측 원점 5개로 다시 채점한 두 모형과 베이스라인

표 1 의 한 주로는 순위를 정하지 않습니다. 원점을 한 주씩 옮겨 5주를 채점하고 투입 판단을 내립니다.

#### 할 일 — 미션 1 (25분)

1. 아래 「미션 1 준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 「만들 것」대로 「PROMPT 셀」의 PROMPT 에 시킬 일을 내 말로 적습니다.
3. 셀을 클릭한 채 AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 합니다. LMS 밖이면 받은 코드를 표시선 아래에 둡니다.
4. 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고쳐 3번부터 다시 합니다.

**이 미션의 질문**: 예측 원점 5개로 채점하면 계절 나이브 168 보다 작게 빗나가는 모형이 있을까요?

<img src="https://skforecast.org/latest/img/backtesting_refit.gif" alt="학습 구간이 늘고 채점 구간이 뒤따르는 애니메이션" width="640" height="284"/>
<p align="center"><em>▲ 예측 원점을 옮겨 가며 다시 채점하는 방법. 초록 학습 구간이 늘어날 때마다 다시 적합하고, 파란 채점 구간이 바로 뒤를 따라간다 (출처: skforecast)</em></p>

**쓸 수 있는 것**: 「미션 1 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `df` · `df['demand']` | 시간당 기록 표와 보정 수요(대) |
| `ORIGINS` · `FIT_START` | 예측 원점 5개(금요일 23시) · 학습 시작 9월 1일 0시 |
| `ORDERS` · `fit_sarimax(y, seasonal_order)` | 두 모형의 계절 차수 · 차수 (1, 0, 1) 과 계절 차수로 적합하는 함수 |

#### 만들 것

1. **셋째 날 SARIMA 5주 평균 RMSE**: 단위 대 · 소수 둘째 자리
   - 원점마다 9월 1일부터 원점까지로 다시 적합해 원점 뒤 168시간을 한 번에 예측하고, RMSE 5개를 평균합니다
2. **계절 나이브 168 5주 평균 RMSE**: 단위 대 · 소수 둘째 자리
   - 같은 원점·같은 168시간에서 1주일 전 같은 시각 값을 예측으로 씁니다
3. **그래프 한 장**: 가로축은 「예측 원점」 1～5, 세로축은 「그 주의 RMSE (대)」인 선 그래프
   - 선 3개(둘째 날 ARIMA · 셋째 날 SARIMA · 계절 나이브 168), 선마다 점 5개. 원점마다 세 값을 한 행씩 출력합니다

**통과 기준**: 1·2번이 「이름 = 값 대」로 한 행씩 출력되고 3번 그래프가 한 장 있으면 통과입니다. 적합을 10번 해서 1～2분 걸립니다. 원점 5 의 세 값은 표 1 과 같아야 합니다.

#### 프롬프트에 넣을 것(힌트)

- **고정할 이름**: 위 표의 이름과 설치 셀의 `rmse(a, b)` 는 다시 만들지 말라고 적습니다.
- **원점마다 할 일**: 학습은 9월 1일부터 원점까지, 채점은 원점 1시간 뒤부터 168시간이라고 적습니다.
- **출력과 그래프**: 값 두 개의 이름·단위·자릿수와 3번의 선 개수, 가로축·세로축 이름을 적습니다.

<details class="lms-extra"><summary>자세히 — 예시 프롬프트</summary>

위 표의 이름과 rmse 는 새로 만들지 말아 줘. 원점마다 FIT_START 부터 원점까지로 두 모형을 다시 적합해 원점 뒤 168시간 RMSE 를 구하고, 같은 168시간의 df['demand'].shift(168) RMSE 도 구해 원점마다 한 행씩 찍어 줘. 세 모형의 5주 평균을 "셋째 날 SARIMA 5주 평균 RMSE = 값 대" 형태로 소수 둘째 자리까지 출력해 줘. 원점별 RMSE 를 선 3개 그래프 한 장으로 그리고 가로축·세로축 이름을 달아 줘.

</details>

#### 예상 적기

> **[예측] 표 1 의 한 주에서는 둘째 날 ARIMA 가 셋째 날 SARIMA 보다 작게 빗나갔습니다. 원점 5개로 채점하면 어떻게 될까요?**
>
> 1. 5개 평균에서도 둘째 날 ARIMA 가 작다. 한 주의 순위가 이어진다
> 2. 평균은 셋째 날 SARIMA 가 작지만, 두 모형 모두 계절 나이브 168 보다 크다
> 3. 두 모형 모두 계절 나이브 168 보다 작아진다. 여러 주를 보면 모형이 앞선다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 이름을 직접 만듭니다
URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")        # 8,760행 x 14열
#  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일창에 올린 뒤
#  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",      # 위치 기반 리네임
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")    # 01/12/2017 = 2017년 12월 1일
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"                       # 등간격 1시간 — 시차 24 가 "24시간 전" 이라는 뜻이 된다
_s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
_y = _s.fillna(_s.shift(24 * 7))          # 1주일 전 같은 시각 값으로 채운다 (요일 효과를 지키려고)
df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로

FIT_START = "2018-09-01 00:00"            # 학습 구간의 시작 — 예측 원점이 옮겨 가도 그대로 둔다
ORIGINS = pd.date_range("2018-10-19 23:00", "2018-11-16 23:00", freq="7D")   # 원점 5개, 모두 금요일 23시
ORDERS = {"둘째 날 ARIMA": (0, 1, 0, 24),                                     # 두 모형의 계절 차수
          "셋째 날 SARIMA": (1, 1, 1, 24)}


def fit_sarimax(y, seasonal_order):
    """차수 (1, 0, 1) 과 계절 차수로 적합합니다. 둘째 날·셋째 날과 같은 적합 조건"""
    return SARIMAX(y, order=(1, 0, 1), seasonal_order=seasonal_order, trend=None,
                   enforce_stationarity=True, enforce_invertibility=True).fit(disp=False)


print(f"행 x 열 : {df.shape[0]:,} x {df.shape[1]}   기간 {df.index.min()} ~ {df.index.max()}")
print("예측 원점 5개: 원점 뒤 168시간이 그 원점의 채점 구간입니다")
for _k, _o in enumerate(ORIGINS, 1):
    _n = len(df.loc[FIT_START:_o])
    print(f"  원점 {_k}  {_o:%m-%d(%a) %H시}   학습 {_n:,}시간"
          f"   채점 {_o + pd.Timedelta(hours=1):%m-%d %H시} ~ {_o + pd.Timedelta(hours=168):%m-%d %H시}")
print("쓸 수 있는 이름 : df (표 전체) · ORIGINS (예측 원점 5개) · FIT_START (학습 시작)"
      " · ORDERS (두 모형의 계절 차수) · fit_sarimax(y, seasonal_order) (적합 함수) · rmse(a, b)")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

## 미션 2 — 학습 폴드 안에서 전처리를 끝낸 교차 검증

분할을 시간 순서로 지켜도 스케일러를 전체 기간으로 fit 하면 미래의 평균이 학습에 들어갑니다. 미션 1 과 같은 순서로 합니다.

**이 미션의 질문**: 분할 1 스케일러의 기온 평균은 전체 평균과 얼마나 다를까요?

모형은 필수 머신러닝에서 본 **릿지 회귀(ridge regression)** 이고, 계수에 벌점을 주므로 표준화를 함께 씁니다. 채점하는 주의 실제 기온·강수를 받는 모형이라 계절 나이브 168 과 나란히 적지 않습니다.

**쓸 수 있는 것**: 「미션 2 준비 셀」이 만들어 둔 이름입니다.

| 이름 | 무엇 |
|---|---|
| `X_cv` · `y_cv` | 2,016시간의 설명 변수 33열(시각 더미 24 · 요일 더미 7 · 기온 · 강수)과 보정 수요(대) |
| `TEMP_COL` | `X_cv` 에서 기온 열의 번호 |
| `TimeSeriesSplit` · `make_pipeline` · `StandardScaler` · `Ridge` | 분할기 · 스케일러와 모형을 묶어 fit 하는 함수 · 스케일러 · 릿지 회귀 |

#### 만들 것

1. **릿지 5분할 평균 RMSE**: 단위 대 · 소수 둘째 자리
   - `TimeSeriesSplit(n_splits=5, test_size=168)` 로 나눕니다. 분할마다 스케일러와 `Ridge(alpha=100)` 을 `make_pipeline` 으로 묶어 학습 폴드로만 fit 하고, 검증 폴드 RMSE 5개를 평균합니다
2. **분할 1 스케일러 기온 평균**: 단위 °C · 소수 둘째 자리
   - 분할 1 은 첫 번째 분할(9월 1일～10월 19일 1,176시간으로 학습)입니다. 이 분할 파이프라인의 스케일러가 계산한 `TEMP_COL` 열 평균(`mean_`)입니다
3. **그래프 한 장**: 가로축은 「분할」 1～5, 세로축은 「검증 폴드 RMSE (대)」인 막대 5개 그래프

**통과 기준**: 1·2번이 「이름 = 값 단위」로 한 행씩 출력되고 막대 그래프가 한 장 있으면 통과입니다.

#### 프롬프트에 넣을 것(힌트)

- **분할기 설정**: `n_splits` 와 `test_size` 를 둘 다 적습니다.
- **fit 하는 행**: 분할마다 학습 폴드로만 fit 하라고 적습니다.
- **꺼낼 값**: 분할 1 스케일러의 `TEMP_COL` 열 평균을 꺼내라고 적습니다. 파이썬은 0부터 세므로 「첫 번째 분할(인덱스 0)」이라고 적습니다.

<img src="https://scikit-learn.org/stable/_images/sphx_glr_plot_cv_indices_013.png" alt="TimeSeriesSplit 분할 그림" width="560" height="280"/>
<p align="center"><em>▲ TimeSeriesSplit 의 분할. 검증(주황)은 언제나 학습(파랑)의 오른쪽이고, 스케일러도 파란 구간으로만 fit 한다 (출처: scikit-learn)</em></p>

#### 예상 적기

분할 1 은 9월 1일～10월 19일로 학습하고 10월 20～26일을 채점합니다. 분할 5 는 9월 1일～11월 16일로 학습합니다.

> **[예측] 분할 1 스케일러의 기온 평균은 전체 2,016시간의 평균과 비교해 어떨까요?**
>
> 1. 더 높다. 분할 1 의 학습 폴드에 추운 11월이 없다
> 2. 같다. 어느 분할에서나 같은 평균을 쓴다
> 3. 더 낮다. 학습 폴드가 짧아 추운 날의 비중이 크다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 이름을 직접 만듭니다
from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

URL = "https://archive.ics.uci.edu/static/public/560/seoul+bike+sharing+demand.zip"
raw = pd.read_csv(URL, encoding="cp1252", compression="zip")        # 8,760행 x 14열
#  받아지지 않으면: UCI 페이지에서 zip 을 내려받아 Colab 좌측 파일창에 올린 뒤
#  URL 대신 "seoul+bike+sharing+demand.zip" 을 넣으세요
raw.columns = ["date", "count", "hour", "temp", "humidity", "wind", "visibility",      # 위치 기반 리네임
               "dewpoint", "solar", "rain", "snow", "season", "holiday", "functioning"]
raw["datetime"] = (pd.to_datetime(raw["date"], format="%d/%m/%Y")    # 01/12/2017 = 2017년 12월 1일
                   + pd.to_timedelta(raw["hour"], unit="h"))
df = raw.sort_values("datetime").set_index("datetime")
df.index.freq = "h"                       # 등간격 1시간
_s = df["count"].astype(float).mask(df["functioning"].eq("No"))   # 휴무는 0 이 아니라 관측 없음
_y = _s.fillna(_s.shift(24 * 7))          # 1주일 전 같은 시각 값으로 채운다
df["demand"] = _y.fillna(_y.shift(-24 * 7))   # 1주일 전도 휴무인 시각만 1주일 뒤 값으로

d_cv = df.loc["2018-09-01 00:00":"2018-11-23 23:00"]                 # 교차 검증에 쓰는 2,016시간
_X = pd.get_dummies(pd.DataFrame({"hour": d_cv.index.hour.astype(str),
                                  "dow": d_cv.index.dayofweek.astype(str)}, index=d_cv.index))
_X["temp"] = d_cv["temp"].values          # 기온 (°C)
_X["rain"] = d_cv["rain"].values          # 강수 (mm)
TEMP_COL = list(_X.columns).index("temp")   # X_cv 에서 기온 열의 번호
X_cv = _X.astype(float).values            # 시각 더미 24 · 요일 더미 7 · 기온 · 강수 = 33열
y_cv = d_cv["demand"].values              # 보정 수요 (대)

print(f"교차 검증 구간 : {d_cv.index.min():%Y-%m-%d %H시} ~ {d_cv.index.max():%Y-%m-%d %H시}"
      f"   ({len(d_cv):,}시간)   설명 변수 {X_cv.shape[1]}열")
print(f"기온 열 번호 TEMP_COL : {TEMP_COL}   전체 2,016시간의 기온 평균 {X_cv[:, TEMP_COL].mean():.2f} °C")
print("쓸 수 있는 이름 : X_cv (설명 변수) · y_cv (보정 수요, 대) · TEMP_COL (기온 열 번호)"
      " · TimeSeriesSplit · make_pipeline · StandardScaler · Ridge · rmse(a, b)")

In [ ]:
# 미션 2 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → 도우미에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

## 오늘의 결론

<div class="lms-summary">

**정리**

- 예측 원점 5개 평균 RMSE: 셋째 날 SARIMA 약 385대, 둘째 날 ARIMA 약 426대, 계절 나이브 168 363.55대. 11월 16일 한 주에서는 둘째 날 ARIMA 가 더 작았지만, 원점 5개 평균에서는 순위가 반대입니다
- 교차 검증 값은 `TimeSeriesSplit(n_splits=5, test_size=168)` 로 나누고 스케일러를 분할마다 학습 폴드로만 fit 한 값만 보고합니다. 분할 1 의 스케일러 기온 평균은 18.79 °C 로, 11월까지 포함한 전체 평균 14.92 °C 와 다릅니다
- 할 일: 운영 예측은 계절 나이브 168 로 냅니다. 코드는 `df["demand"].shift(168)` 한 행입니다. 평가표에는 원점 5개별 RMSE 와 평균, 학습 구간(9월 1일부터 원점까지), 예측 기간 168시간을 함께 적습니다

두 모형 모두 원점 5개 평균이 베이스라인 363.55 보다 커서 아직 투입하지 않습니다.

</div>

> **[과제] 실습 제출 — 예측 원점 5개 채점과 학습 폴드 안의 교차 검증**
>
> 미션 1·2 의 「PROMPT 셀」이 채점을 통과한 상태로 노트북을 제출합니다. PROMPT 는 내가 쓴 그대로 둡니다.
>
> 마감: 2026-10-13T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 미션 1 | 채점을 통과했고 PROMPT 에 만들 것이 내 말로 적혀 있다 | 50 |
> | 미션 2 | 채점을 통과했고 PROMPT 에 만들 것이 내 말로 적혀 있다 | 50 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 시계열 넷째 날 완료</h2>
<div style="color:#656565">예측 원점 5개에서 두 모형과 베이스라인을 같은 조건으로 채점하고, 학습 폴드 안에서 전처리를 끝낸 교차 검증 값을 만들었습니다.</div>
</div>